# Cross-talk and demixing

Notebook 1 built a recording **forward** and showed that the movie is a sum of
rank-one layers, `movie = A·C + b·f`: each cell `i` contributes its spatial footprint
`A_i` times its calcium trace `C_i`, on top of a diffuse background `b·f`. This
notebook asks the question that motivates the whole analysis pipeline: **given the
movie, how do you read one cell's calcium back out?**

The obvious answer - draw a region of interest around the cell and average its pixels
- is wrong, and *understanding why* is the point. Optical blur and 1-photon tissue
scatter (Notebook 1's optics stage) spread every footprint, so a cell's ROI also
collects light from its **neighbours** (cross-talk) and sits on the **neuropil**
pedestal. The naive trace is a mixture, not the cell.

The fix is **demixing**: solve the linear system `movie = A·C + b·f` for `C` instead
of masking. Because minisim generated the recording, we hold the true `A`, `C`, and
`b` - so we can show the contamination exactly, then show demixing remove it, and
finally find the regime where even *perfect-knowledge* demixing fails.

Three parts:

1. **The problem** - the naive footprint-ROI trace, and an honest accounting of what
   contaminates it.
2. **The fix** - demixing as the inverse of `A·C`, the factorization CNMF performs.
3. **The limit** - when footprints overlap too much, even oracle demixing breaks down.

> Like Notebook 3, this notebook is **static**: each demo shows the whole effect, so it
> reads correctly without a live kernel. Every demo cell starts with a `# try:` knob.

## Setup

We need a recording with genuine cross-talk. The CI fixture `make_recording` places
cells on a well-separated grid *to avoid* overlap - the opposite of what we want here
- so we pack many cells into the field of view: the grid spacing (~14 um here) shrinks
until the blurred, scattered footprints overlap, which is exactly the crowded regime
real cortex presents. We append a `Neuropil` step - the diffuse background glow from the
felt of dendrites and axons between the cells - so the background pedestal is in play
too, and keep the true `A`, `C`, and background fields from `ground_truth`.

Every cell here clears the SNR detection floor (Notebook 1), so all 200 are
**detectable**. Hold onto that: detectability and *recoverability* are different axes.
A cell can be bright enough to detect and still have its trace hopelessly mixed with its
neighbours' - which is exactly the problem this notebook is about. Detection is Notebook
1's subject; this notebook lives on the second axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from minisim.metrics import footprint_mask, footprint_roi_trace, trace_pearson
from minisim.spec import Neuropil, CellActivity
from minisim.testing import make_recording
from minisim.notebooks._support import play, build_naive_overlay_frames

%matplotlib inline
plt.rcParams["image.cmap"] = "magma"
plt.rcParams["figure.dpi"] = 100

# Sparse-ish calcium: each cell fires a handful of well-separated transients over the clip
# (a lower onset rate than make_recording's lively default), the way real GCaMP looks.
ACTIVITY = CellActivity(p_quiescent_to_active=0.02)

# A crowded recording: 200 cells in a 200 um FOV (100 px x 2 um/px) at 100 um depth, with
# neuropil. At this density the grid spacing (~14 um) is comparable to a blurred footprint,
# so footprints overlap and the naive ROI traces are contaminated. We study the `detectable`
# cells (all 200 clear Notebook 1's SNR floor - detection is not the problem here; demixing is).
rec = make_recording(n_cells=200, n_px=100, pixel_size_um=2.0, depth_um=100.0,
                     duration_s=20.0, seed=0, activity=ACTIVITY, extra_steps=[Neuropil()])
gt = rec.ground_truth
movie = np.asarray(rec.observed, float)
A = np.asarray(gt.A_observed)               # (n, H, W) degraded footprints - the spatial model
C = np.asarray(gt.C)                         # (n, frames) true calcium - the answer key
centers = np.asarray(gt.centers_um)          # (n, 3) (z, y, x) soma centers
fps = rec.spec.acquisition.fps
t = np.arange(movie.shape[0]) / fps
print(f"{gt.n_units} cells, {int(np.asarray(gt.detectable).sum())} detectable; "
      f"movie {movie.shape}")

# The image sensor turns intensity into integer counts with a fixed gain
# (QE x photons-per-unit x ADU-per-electron). The part of a cell's naive ROI that is
# genuinely its OWN light is that gain times the mask-mean of its footprint, times C.
_sensor = next(s for s in rec.spec.steps if s.kind == "sensor")
_isensor = rec.spec.acquisition.image_sensor
count_gain = _isensor.quantum_efficiency * _sensor.photons_per_unit * _isensor.gain_adu_per_e


def corr(a, b):
    # Pearson correlation between two 1-D traces (the same metric as the library's
    # trace_pearson, here for ad-hoc pairs; 0 when either trace is flat).
    a = np.asarray(a, float) - np.mean(a); b = np.asarray(b, float) - np.mean(b)
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / d) if d else 0.0


def nearest_others(i, k=3):
    # Indices of the k nearest OTHER cells, by lateral (y, x) distance.
    d = np.linalg.norm(centers[:, 1:] - centers[i, 1:], axis=1)
    return [j for j in np.argsort(d) if j != i][:k]


# How tight the naive ROI hugs each cell: footprint_mask keeps pixels above ROI_REL x peak,
# so a higher value draws a smaller mask on the bright core - where the cell's own light
# dominates and less of the neighbours' dim-edge bleed gets in. The library default is 0.18
# (a generous, eyeballed ROI); we draw a little tighter so each cell's own C is a clearer
# share of its trace, without erasing the contamination that motivates demixing.
ROI_REL = 0.30                                   # try: 0.18 (the default) .. 0.5 (very tight)


def own_roi_light(i):
    # Cell i's own light inside its ROI, in movie counts: gain x (mean of its footprint over
    # the ROI mask) x its true calcium C. Every other contribution (neighbours, neuropil)
    # only adds light, so the naive ROI sits at or above this everywhere.
    g = A[i][footprint_mask(A[i], ROI_REL)].mean()
    return count_gain * g * C[i]

### The recording

Here is the crowded movie we will dissect: 200 cells in a 200 um field, their blurred
footprints overlapping on the neuropil background. Try to tell, by eye, which flicker
belongs to which cell - that difficulty is the whole problem.

In [ ]:
play(movie, fps=fps, height=320, title="the crowded recording (observed counts)")

# Part 1 - the problem: cross-talk

A footprint ROI is what a person draws by eye: the bright pixels of one cell. Averaging
the movie over that mask (`footprint_roi_trace`) is the simplest possible read-out.
This part shows that read-out is a *mixture* and accounts for what is in it.

## 1.1 The naive read-out does not recover the cell

`footprint_roi_trace` averages the movie over a cell's footprint mask - exactly a
hand-drawn ROI, with no unmixing. We compare it against the cell's **own light**: its true
calcium `C` scaled into the ROI by the sensor gain and the footprint's mask-mean, i.e. the
part of the measured trace that genuinely belongs to this cell. Because every other source
(neighbours, neuropil) only *adds* light, the naive ROI sits **on or above** the cell's own
light everywhere - the gap between them is the contamination.

The video below plays the crowded recording (left, three contaminated cells ringed in
colour) next to each cell's naive ROI trace overlaid on its own light, with a cursor marking
the current frame. Watch the naive trace jump on transients the cell never fired: those bumps
are a ringed **neighbour's** events, blurred and scattered into the mask, while the dashed
own-light line stays flat. (Both traces are in the movie's count units, the own light shifted
onto the same background pedestal as the naive ROI so they are comparable.)

In [ ]:
def most_contaminated(n=3, min_sep_um=30.0):
    # Detectable cells whose naive ROI correlates most with a NEIGHBOUR's true C - the
    # clearest victims of cross-talk (typically quiet cells beside busy ones). Pick them
    # greedily so the chosen cells sit at least `min_sep_um` apart and read as distinct
    # coloured rings in the movie.
    det = np.where(np.asarray(gt.detectable))[0]
    score = [max(corr(footprint_roi_trace(movie, A[i], ROI_REL), C[j]) for j in nearest_others(i))
             for i in det]
    picks = []
    for i in det[np.argsort(score)[::-1]]:
        if all(np.linalg.norm(centers[i, 1:] - centers[p, 1:]) > min_sep_um for p in picks):
            picks.append(int(i))
        if len(picks) == n:
            break
    return picks


# A side-by-side video, in the layout of Notebook 1's Stage 3: the crowded movie (left, the
# picked cells ringed in colour) next to each cell's naive ROI trace overlaid on its own
# light, with a vertical cursor marking the current frame. As the cursor sweeps, the naive
# trace jumps on transients the cell never fired - light from a ringed neighbour bleeding into
# the mask - while the dashed own-light line stays flat.
picks = most_contaminated(3)                            # try: pick your own cells, e.g. [5, 12, 30]
colors = ["#ff5050", "#3fa7ff", "#ffd23f"]
naive_traces = np.array([footprint_roi_trace(movie, A[i], ROI_REL) for i in picks])
own_traces = np.array([own_roi_light(i) for i in picks])
own_traces += (naive_traces - own_traces).min(axis=1, keepdims=True)   # share the background pedestal
vmax = float(np.percentile(movie, 99.9))
frames = build_naive_overlay_frames(movie, gt, picks, colors, t, vmax,
                                    rec.spec.acquisition.pixel_size_um, naive_traces, own_traces)
play(frames, fps=fps, height=340,
     title="the naive ROI picks up neighbours the cell never fired")

## 1.2 What is in the naive trace?

Because we hold every cell's true `C` and the background driver, we can *attribute* the
naive ROI to its sources. Regress the naive trace onto the cell's own `C`, its nearest
neighbours' `C`, and the neuropil population driver `P(t)` (the aggregate local activity
that drives the background haze): the fitted pieces add back up to the naive trace, and
each piece is one contamination term:

`naive ROI  ≈  (own C)  +  (neighbour bleed)  +  (neuropil pedestal)  +  offset`.

The own-cell piece is the signal we wanted; everything else is contamination the ROI
cannot separate.

In [ ]:
def attribute_roi(i, k=3):
    # Least-squares attribution of the naive ROI to true sources: own C, k nearest
    # neighbours' C, the neuropil driver P(t), and a constant. Returns each fitted
    # contribution (these sum to the model's fit of the naive trace).
    naive = footprint_roi_trace(movie, A[i], ROI_REL)
    nbrs = nearest_others(i, k)
    P = (np.asarray(gt.neuropil_population, float) if gt.neuropil_population is not None
         else np.ones(movie.shape[0]))
    cols = [C[i]] + [C[j] for j in nbrs] + [P, np.ones_like(P)]
    X = np.array(cols, float).T                       # (frames, 2 + k)
    beta, *_ = np.linalg.lstsq(X, naive, rcond=None)
    contrib = X * beta                                 # per-source fitted contribution
    own = contrib[:, 0]
    neigh = contrib[:, 1:1 + k].sum(axis=1)
    pedestal = contrib[:, 1 + k]
    return naive, own, neigh, pedestal, nbrs


def show_contamination_anatomy():
    # try: change the cell, or k (number of neighbours attributed).
    i = most_contaminated(1)[0]
    naive, own, neigh, pedestal, nbrs = attribute_roi(i)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(t, naive, color="k", lw=1.4, label="naive ROI (measured)")
    ax.plot(t, own, color="C0", lw=1.0, label="own C (the signal)")
    ax.plot(t, neigh, color="C3", lw=1.0, label=f"neighbour bleed ({len(nbrs)} nearest)")
    ax.plot(t, pedestal - pedestal.mean(), color="C2", lw=1.0, label="neuropil pedestal")
    ax.set(xlabel="time (s)", ylabel="ROI fluorescence (a.u.)",
           title=f"cell {i}: the naive ROI decomposed into its sources")
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    plt.show()


show_contamination_anatomy()

Across the whole detectable population, the cross-talk has a clear **spatial
signature**. Take every *pair* of cells, measure how correlated their two traces are
(**Pearson correlation**: +1 = identical shape, 0 = unrelated, regardless of scale), and
plot that against the distance between the pair.

The **true calcium** `C` (right) is flat near 0 at every distance - real neurons fire
independently of how close they sit. The **naive ROI** traces (left) tell a different
story: nearby pairs are strongly correlated, and that correlation decays to 0 as the pair
separates. That rise at short range is pure cross-talk - two neighbouring masks sharing
each other's blurred, scattered light. The gap between the two panels is the
contamination, and the distance over which it dies off is the footprint overlap the
optics set.

In [ ]:
def show_pairwise_corr_vs_distance():
    # try: sweep depth_um or n_cells in Setup and watch the naive curve's range grow.
    det = np.where(np.asarray(gt.detectable))[0]
    naive = np.array([footprint_roi_trace(movie, A[i], ROI_REL) for i in det])
    pos = centers[det, 1:]                            # lateral (y, x) of each cell
    pairs = [(a, b) for a in range(len(det)) for b in range(a + 1, len(det))]
    dist = np.array([np.linalg.norm(pos[a] - pos[b]) for a, b in pairs])
    c_naive = np.array([corr(naive[a], naive[b]) for a, b in pairs])
    c_true = np.array([corr(C[det[a]], C[det[b]]) for a, b in pairs])

    def binned(c, edges):
        ctr = 0.5 * (edges[:-1] + edges[1:])
        m = [c[(dist >= lo) & (dist < hi)].mean() if ((dist >= lo) & (dist < hi)).any()
             else np.nan for lo, hi in zip(edges[:-1], edges[1:])]
        return ctr, np.array(m)

    edges = np.linspace(0, dist.max(), 9)
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 4.3), sharex=True, sharey=True)
    for ax, c, ttl, col in [(axl, c_naive, "naive ROI traces", "C1"),
                            (axr, c_true, "true C (the answer key)", "0.4")]:
        ax.scatter(dist, c, s=9, alpha=0.25, color=col)
        ctr, m = binned(c, edges)
        ax.plot(ctr, m, color="C3", lw=2.2, marker="o", label="binned mean")
        ax.axhline(0, color="0.7", lw=0.8, ls="--")
        ax.set(xlabel="distance between cell pair (um)", title=ttl, ylim=(-0.3, 1.02))
        ax.legend(fontsize=8, loc="upper right")
    axl.set_ylabel("pairwise trace correlation")
    fig.suptitle("cross-talk is local: nearby naive traces correlate; the true C does not")
    fig.tight_layout()
    plt.show()


show_pairwise_corr_vs_distance()

## 1.3 Why: the footprints physically overlap

Cross-talk is not a measurement mistake - it is in the data. Notebook 1's optics stage
blurred each footprint (diffraction + defocus) and the tissue scattered it; at this depth
and density the footprints overlap, so one cell's ROI mask sits on top of its neighbours'
light. To see it directly, crop the region around one cell and compare, on the same colour
scale: **its own footprint** versus **every other cell's footprint summed**. Inside the
outlined ROI mask the neighbours' light is not a faint halo - it is a large fraction of the
total, which is exactly why the naive ROI is contaminated.

In [ ]:
def show_footprint_overlap():
    # try: outline a different cell's ROI mask, or change the crop padding.
    i = most_contaminated(1)[0]
    mask = footprint_mask(A[i], ROI_REL)
    others = A.sum(axis=0) - A[i]                      # every OTHER cell's footprint, summed
    frac = others[mask].sum() / (A[i][mask].sum() + others[mask].sum())
    ys, xs = np.where(mask); pad = 8
    y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad, mask.shape[0])
    x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad, mask.shape[1])
    vmax = max(A[i][y0:y1, x0:x1].max(), others[y0:y1, x0:x1].max())
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 4.6))
    for ax, img, ttl in [(axl, A[i], f"cell {i} alone (its own footprint)"),
                         (axr, others, "every other cell, summed")]:
        ax.imshow(img[y0:y1, x0:x1], vmin=0, vmax=vmax)
        ax.contour(mask[y0:y1, x0:x1], levels=[0.5], colors="w", linewidths=1.4)
        ax.set_title(ttl); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"inside cell {i}'s ROI mask, {frac:.0%} of the footprint light is its neighbours'")
    plt.show()


show_footprint_overlap()

# Part 2 - the fix: demixing

The naive ROI fails because masking ignores the spatial model. Demixing uses it: the
movie *is* `A·C + b·f`, a linear system, so given the spatial maps we can solve for the
traces instead of averaging pixels.

## 2.1 Solve the system instead of masking

Stack every known spatial map into one matrix `D` - one column per cell footprint
`A_i`, one per neuropil field `b_k`, and a constant for any flat offset (leakage). Each
frame of the movie is then `movie[t] = D · x[t]`, and least squares recovers `x[t]`:
the first rows are the cells' demixed traces, the rest the background.

This is an **oracle** demix - it uses the *true* footprints, the most any method could
know. A real pipeline (CNMF) must also *discover* `A`; here we isolate the demixing
step alone, to see what it can and cannot do even with perfect spatial knowledge.

In [ ]:
def design_matrix(A, neuropil_spatial, H, W):
    # The known spatial maps stacked as columns: one per cell footprint, one per neuropil
    # field, and a constant (flat offset / leakage). Demixing solves the movie against these.
    n = A.shape[0]
    cols = [A.reshape(n, -1)]
    if neuropil_spatial is not None:
        cols.append(np.asarray(neuropil_spatial).reshape(-1, H * W))
    cols.append(np.ones((1, H * W)))                 # constant (leakage / offset)
    return np.concatenate(cols, axis=0).T            # (HW, n + K + 1)


def oracle_demix(movie, A, neuropil_spatial):
    # Solve movie[t] = D @ x[t] for all frames at once; lstsq returns x, whose first
    # n rows are the demixed cell traces (in the movie's count units).
    T, H, W = movie.shape
    D = design_matrix(A, neuropil_spatial, H, W)
    X, *_ = np.linalg.lstsq(D, movie.reshape(T, -1).T, rcond=None)
    return X[:A.shape[0]]                            # (n, frames) demixed cell traces


C_demix = oracle_demix(movie, A, gt.neuropil_spatial)


def show_demix_vs_naive():
    # try: pick your own cells. Traces are the cell's own light in the ROI (Section 1.1):
    # true and demixed should overlap; the naive sits above them by the contamination.
    picks = most_contaminated(3)
    fig, axes = plt.subplots(len(picks), 1, figsize=(9, 5.8), sharex=True)
    for ax, i in zip(np.atleast_1d(axes), picks):
        naive = footprint_roi_trace(movie, A[i], ROI_REL)
        g = A[i][footprint_mask(A[i], ROI_REL)].mean()
        true_own = own_roi_light(i)                  # gain * g * C
        demix_own = g * C_demix[i]                   # C_demix already carries the sensor gain
        base = (naive - true_own).min()              # share the background pedestal
        ax.plot(t, naive, color="C1", lw=0.8, label="naive ROI")
        ax.plot(t, demix_own + base, color="C2", lw=1.3, label="demixed")
        ax.plot(t, true_own + base, color="k", lw=1.0, ls="--", label="true (own light)")
        ax.set_ylabel(f"cell {i}"); ax.set_yticks([])
        ax.legend(fontsize=7, ncol=3, loc="upper right")
    np.atleast_1d(axes)[-1].set_xlabel("time (s)")
    np.atleast_1d(axes)[0].set_title("demixing recovers the cell's own light where the naive ROI could not")
    plt.show()


show_demix_vs_naive()

## 2.2 Score it: contamination gone, traces recovered

The fix is not just visual. For every detectable cell, compare the naive ROI and the demixed
trace on two scores from Notebook 3: the trace correlation with the true `C` (higher is
better), and the correlation with the nearest neighbour's `C` (the cross-talk - lower is
better). Demixing pushes the own-correlation up (here about 0.73 to 0.97) and the
neighbour-correlation sharply down (about 0.50 to 0.13) - not all the way to zero, since
heavily overlapping footprints leave a little residual cross-talk, which is the limit Part 3
is about.

In [ ]:
def show_recovery_scores():
    # try: restrict det to the deepest cells (e.g. centers[det, 0] > 100) to score the hardest ones.
    det = np.where(np.asarray(gt.detectable))[0]
    naive_stack = np.array([footprint_roi_trace(movie, A[i], ROI_REL) for i in det])
    own_naive = trace_pearson(naive_stack, C[det], [(k, k) for k in range(len(det))])
    own_demix = trace_pearson(C_demix[det], C[det], [(k, k) for k in range(len(det))])
    nbr_naive = np.array([max(corr(naive_stack[k], C[j]) for j in nearest_others(i))
                          for k, i in enumerate(det)])
    nbr_demix = np.array([max(corr(C_demix[i], C[j]) for j in nearest_others(i)) for i in det])
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, naive_v, demix_v, ttl in [
        (axl, own_naive, own_demix, "correlation with own C (higher = better)"),
        (axr, nbr_naive, nbr_demix, "correlation with neighbour C (lower = better)"),
    ]:
        ax.hist(naive_v, bins=np.linspace(-0.2, 1, 25), alpha=0.6, color="C1", label="naive ROI")
        ax.hist(demix_v, bins=np.linspace(-0.2, 1, 25), alpha=0.6, color="C2", label="demixed")
        ax.set(title=ttl, xlabel="correlation", ylabel="cells"); ax.legend(fontsize=8)
    plt.show()
    print(f"median own-C corr   naive {np.nanmedian(own_naive):.2f}  ->  demix {np.nanmedian(own_demix):.2f}")
    print(f"median neighbour corr naive {np.median(nbr_naive):.2f}  ->  demix {np.median(nbr_demix):.2f}")


show_recovery_scores()

## 2.3 This is what CNMF does

The solve above *is* the `A·C` factorization run backward. CNMF (minian's core)
estimates the same `A` and `C` from the movie - the difference is that it has no oracle:
it must **discover** the footprints `A` and the background `b` at the same time as the
traces, alternating between updating the spatial maps and the temporal ones. Our oracle
demix is the best case that bounds it: it shows that *once the spatial model is right*,
the cross-talk is removable. So how well a real pipeline does comes down to how close its
discovered `A` is to the truth - and the gap between this oracle ceiling and a real run
is exactly what Notebook 3 measures.

So the forward chain of Notebook 1 and the inverse here meet: `A·C + b·f` built up from
physics, and `A·C + b·f` taken apart to read the biology back out.

# Part 3 - the limit: when even oracle demixing fails

Demixing is a linear solve, and how hard a linear solve is to invert is captured by one
number, its **condition number** (`cond(D)`): low means the columns of `D` point in
clearly different directions and the solve is stable; high means some columns are nearly
parallel, so tiny bits of noise blow up into large errors in the answer. When footprints
overlap only a little, their columns in `D` are nearly independent and `cond(D)` is low.
As cells crowd closer (higher density, or deeper cells with more scatter), the columns
become nearly parallel - two footprints the optics can no longer tell apart - `cond(D)`
explodes, and the system becomes ill-conditioned. Past that point, no demixer, however
perfect its knowledge of `A`, can separate the traces: the information is gone from the
movie.

Sweep density (cells in the same FOV) and watch both read-outs against `cond(D)`. The
naive ROI degrades throughout; oracle demixing holds near-perfect while `cond(D)` stays
low, then collapses as `cond(D)` explodes - the irreducible limit any pipeline inherits.
Note the `N/N detectable` labels: every cell stays above the detection floor the whole
way, yet the traces become unrecoverable - detectability is not recoverability.

In [ ]:
def show_density_limit():
    # try: change the densities, or sweep depth_um instead of n_cells. One short
    # recording per point (kept brief so the sweep stays quick).
    counts = [40, 90, 160, 260, 400]
    naive_r, demix_r, cond_d, n_det = [], [], [], []
    for n in counts:
        r = make_recording(n_cells=n, n_px=100, pixel_size_um=2.0, depth_um=120.0,
                           duration_s=12.0, seed=0, activity=ACTIVITY, extra_steps=[Neuropil()])
        g = r.ground_truth; mv = np.asarray(r.observed, float)
        Aa = np.asarray(g.A_observed); Cc = np.asarray(g.C); H, W = mv.shape[1:]
        det = np.where(np.asarray(g.detectable))[0]
        # The same least-squares demix as Section 2.1, here solved via the normal equations
        # (D.T @ D) x = D.T @ movie - much faster across a whole sweep, and identical in this
        # regime (cond(D) stays well below where that shortcut would lose accuracy). The
        # condition number falls straight out of the same D.T @ D.
        D = design_matrix(Aa, g.neuropil_spatial, H, W)
        G = D.T @ D
        Cd = np.linalg.solve(G, D.T @ mv.reshape(mv.shape[0], -1).T)[:n]
        ev = np.linalg.eigvalsh(G)
        cond_d.append(float(np.sqrt(ev[-1] / max(ev[0], 1e-12))))
        pr = [(k, k) for k in range(len(det))]
        naive_stack = np.array([footprint_roi_trace(mv, Aa[i], ROI_REL) for i in det])
        naive_r.append(float(np.nanmedian(trace_pearson(naive_stack, Cc[det], pr))))
        demix_r.append(float(np.nanmedian(trace_pearson(Cd[det], Cc[det], pr))))
        n_det.append(len(det))
    fig, ax = plt.subplots(figsize=(7.6, 4.4))
    ax.plot(counts, naive_r, marker="o", color="C1", label="naive ROI")
    ax.plot(counts, demix_r, marker="s", color="C2", label="oracle demix")
    ax.set(xlabel="cells in the 200 um FOV (density)", ylabel="median corr with true C",
           title="oracle demixing holds, then collapses as cond(D) explodes", ylim=(0, 1.02))
    for x, y, nd, n in zip(counts, demix_r, n_det, counts):
        ax.annotate(f"{nd}/{n} det", (x, y), textcoords="offset points", xytext=(0, 8), fontsize=7)
    axc = ax.twinx()                                  # condition number of D, log scale
    axc.plot(counts, cond_d, marker="^", color="0.4", ls=":", label="cond(D)")
    axc.set_yscale("log"); axc.set_ylabel("cond(D), log scale", color="0.4")
    axc.tick_params(axis="y", labelcolor="0.4")
    ax.legend(loc="center left"); axc.legend(loc="lower right")
    ax.grid(alpha=0.3)
    plt.show()


show_density_limit()

## Recap

- A recording is `A·C + b·f`. Reading a cell out by **averaging an ROI** does not
  recover its calcium: optical blur and tissue scatter spread every footprint, so the
  mask also collects **neighbour bleed** and the **neuropil pedestal** (Part 1).
- **Demixing** - solving `movie = A·C + b·f` for the traces instead of masking -
  removes the cross-talk, recovering `C` where the naive ROI could not (Part 2). This is
  the `A·C` factorization **CNMF** performs, run as the inverse of Notebook 1's forward
  build; the difference is that CNMF must also *discover* `A`.
- Demixing has an **irreducible limit**: when footprints overlap too much, the linear
  system is ill-conditioned (`cond(D)` explodes) and even oracle demixing - perfect
  knowledge of `A` - cannot separate the traces (Part 3). That limit is physics, set by
  the optics and the density, and it bounds every analysis pipeline.
- **Detectable is not recoverable.** Every cell in these recordings clears the SNR
  detection floor, yet at high overlap their traces still cannot be separated: detection
  (Notebook 1) and demixing are different axes, and passing the first does not guarantee
  the second.

With the three notebooks together: Notebook 1 builds the recording forward from physics,
this notebook inverts it to read the biology back out, and Notebook 3 scores how well a
real pipeline does the same.